# Create MITRE ATT&CK Tactics & Techniques json files

Download MITRE ATT&CK taxonomy data (most recent **enterprise-attack**, **ics-attack** & **mobile-attack** json files) from: https://github.com/mitre-attack/attack-stix-data.

Make sure the downloaded taxonomy data files are available in the mitre_attack_files folder.

In [1]:
from mitreattack.stix20 import MitreAttackData
import json

In [2]:
def get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique_stix_id):

    # get data components detecting technique
    datacomponents_detects_technique = mitre_attack_data.get_datacomponents_detecting_technique(technique_stix_id)

    # Get all data sources and data components.
    data_sources = []
    data_components = []
    for d in datacomponents_detects_technique:
        datacomponent = d["object"]
        datasource = mitre_attack_data.get_object_by_stix_id(datacomponent.x_mitre_data_source_ref)
        data_sources.append(datasource.name)
        data_components.append(datacomponent.name)

    # Remove duplicates in the data sources.
    data_sources = list(set(data_sources))

    return data_sources, data_components

In [9]:
def process_mitre_attack_json(json_file_path):
    """Process the ATT&CK data from a single ATT&CK json file."""
    
    # Initialize the MitreAttackData class
    mitre_attack_data = MitreAttackData(json_file_path)
    
    # Fetch all tactics
    tactics = mitre_attack_data.get_tactics_by_matrix()
    
    # Prepare the structure for the JSON
    tactics_list = []

    # We assume that our ATT&CK JSON files contain a single domain, which we extract here.
    if 'Enterprise ATT&CK' in list(tactics.keys()):
        domain_str = 'Enterprise ATT&CK'
        domain = 'enterprise-attack'
    # We ignore the 'Network-Based Effects' key that appears in the tactics.keys() for the Mobile domain.
    if 'Mobile ATT&CK' in list(tactics.keys()):
        domain_str = 'Mobile ATT&CK'
        domain = 'mobile-attack'
    if 'ATT&CK for ICS' in list(tactics.keys()):
        domain_str = 'ATT&CK for ICS'
        domain = 'ics-attack'
    
    # Iterate over each tactic
    for tactic in tactics[domain_str]:
        # Fetch all techniques for the current tactic
        techniques = mitre_attack_data.get_techniques_by_tactic(tactic['x_mitre_shortname'], domain, remove_revoked_deprecated=True)

        # Prepare the techniques list with sub-techniques nested under their parent techniques
        techniques_dict = {}
        for technique in techniques:

            # # Skip deprecated and revoked techniques.
            # if (technique.get('x_mitre_deprecated') == True) or (technique.get('revoked') == True):
            #     print('test')
            #     continue

            # # Process data sources strings into separate 'data sources' and 'data components' labels.
            # data_sources_string = technique.get('x_mitre_data_sources')
            # data_sources = []
            # data_components = []
            # if data_sources_string != None:
            #     for x in data_sources_string:
            #         data_source, data_component = x.split(': ')
            #         data_sources.append(data_source)
            #         data_components.append(data_component)
            #     data_sources = list(set(data_sources))
            #     data_components = list(set(data_components))

            data_sources, data_components = get_data_sources_and_components_by_technique_stix_id(mitre_attack_data, technique.id)
            groups = mitre_attack_data.get_groups_using_technique(technique.id)
            groups = [x['object']['name'] for x in groups]
            occurrence = len(groups)
            
            technique_entry = {
                "name": technique['name'],
                "external_id": technique['external_references'][0]['external_id'],
                "platforms": technique.get('x_mitre_platforms'),
                'groups': groups,
                'occurrence': occurrence,
                'data_sources': data_sources,
                'data_components': data_components,
                "visibility": False,  # Set default visibility to False, can be modified as needed
                "alpha": 0,
            }
            if technique.get('x_mitre_is_subtechnique'):
                parent_id = technique['external_references'][0]['external_id'].split('.')[0]
                if parent_id in techniques_dict:
                    if 'sub_techniques' not in techniques_dict[parent_id]:
                        techniques_dict[parent_id]['sub_techniques'] = []
                    techniques_dict[parent_id]['sub_techniques'].append(technique_entry)
                else:
                    techniques_dict[parent_id] = {
                        "sub_techniques": [technique_entry]
                    }
            else:
                techniques_dict[technique_entry['external_id']] = technique_entry
    
        # Sort the techniques alphabetically by name
        techniques_list = sorted([value for value in techniques_dict.values() if 'name' in value], key=lambda x: x['name'])
    
        # Sort the sub-techniques alphabetically by name
        for technique in techniques_list:
            if 'sub_techniques' in technique:
                technique['sub_techniques'] = sorted(technique['sub_techniques'], key=lambda x: x['name'])
    
        # Prepare the tactic entry
        tactic_entry = {
            "name": tactic['name'],
            "external_id": tactic['external_references'][0]['external_id'],
            "techniques": techniques_list
        }
    
        # Add the tactic entry to the tactics list
        tactics_list.append(tactic_entry)

    return tactics_list

In [10]:
def flatten(lst):
    """
    Flattens a nested list, ignoring NoneType objects.

    Parameters:
    lst (list): A list that may contain nested lists and NoneType objects.

    Returns:
    list: A flattened list with NoneType objects removed.
    """
    result = []

    def _flatten(sublist):
        for item in sublist:
            if item is None:
                continue
            if isinstance(item, list):
                _flatten(item)
            else:
                result.append(item)

    _flatten(lst)
    return result

In [11]:
def get_tactics_metadata(tactics):

    platforms = []
    data_sources = []
    data_components = []
    
    for tactic in tactics:
        for technique in tactic['techniques']:
            platforms.append(technique.get('platforms'))
            data_sources.append(technique.get('data_sources'))
            data_components.append(technique.get('data_components'))
            if 'sub_technique' in technique.keys():
                for sub_technique in technique['sub_techniques']:
                    platforms.append(sub_technique.get('platforms'))
                    data_sources.append(sub_technique.get('data_sources'))
                    data_components.append(sub_technique.get('data_components'))
    
    platforms = list(set(flatten(platforms)))
    data_sources = list(set(flatten(data_sources)))
    data_components = list(set(flatten(data_components)))

    return platforms, data_sources, data_components

In [12]:
def convert_list(attribute_list):
    return [{"name": name, "active": active} for name, active in list(zip(attribute_list, [True]*len(attribute_list)))]

In [13]:
# Load JSON files of all ATT&CK domains.
enterprise_tactics = process_mitre_attack_json("mitre_attack_files/enterprise-attack-15.1.json")
ics_tactics = process_mitre_attack_json("mitre_attack_files/ics-attack-15.1.json")
mobile_tactics = process_mitre_attack_json("mitre_attack_files/mobile-attack-15.1.json")

enterprise_platforms, enterprise_data_sources, enterprise_data_components = get_tactics_metadata(enterprise_tactics)
ics_platforms, ics_data_sources, ics_data_components = get_tactics_metadata(ics_tactics)
mobile_platforms, mobile_data_sources, mobile_data_components = get_tactics_metadata(mobile_tactics)



# Prepare the final JSON structure.
final_json = {
    "enterprise": {
        "tactics": enterprise_tactics,
        "platforms": convert_list(enterprise_platforms),
        "data_sources": convert_list(enterprise_data_sources),
        "data_components": convert_list(enterprise_data_components)
    },
    "ics": {
        "tactics": ics_tactics,
        "platforms": convert_list(ics_platforms),
        "data_sources": convert_list(ics_data_sources),
        "data_components": convert_list(ics_data_components)
    },
    "mobile": {
        "tactics": mobile_tactics,
        "platforms": convert_list(mobile_platforms),
        "data_sources": convert_list(mobile_data_sources),
        "data_components": convert_list(mobile_data_components)
    }
}
    
# Write the JSON to a file.
with open("tactics_and_techniques_by_domain.json", "w") as json_file:
    json.dump(final_json, json_file, indent=2)

print("JSON file created successfully!")

JSON file created successfully!


In [65]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-15.1.json")

# get campaigns related to T1049
technique_stix_id = "attack-pattern--7e150503-88e7-4861-866b-ff1ac82c4475"
campaigns_using_t1049 = mitre_attack_data.get_campaigns_using_technique(technique_stix_id)

print(f"Campaigns using T1049 ({len(campaigns_using_t1049)}):")
for c in campaigns_using_t1049:
    campaign = c["object"]
    print(f"* {campaign.name} ({mitre_attack_data.get_attack_id(campaign.id)})")

Campaigns using T1049 (3):
* Operation Wocao (C0014)
* Operation CuckooBees (C0012)
* FunnyDream (C0007)


In [14]:
mitre_attack_data = MitreAttackData("mitre_attack_files/enterprise-attack-15.1.json")

# get groups related to T1014
technique_stix_id = "attack-pattern--0f20e3cb-245b-4a61-8a91-2d93f7cb0e9b"
groups_using_t1014 = mitre_attack_data.get_groups_using_technique(technique_stix_id)

print(f"Groups using T1014 ({len(groups_using_t1014)}):")
for g in groups_using_t1014:
    group = g["object"]
    print(f"* {group.name} ({mitre_attack_data.get_attack_id(group.id)})")

Groups using T1014 (5):
* Winnti Group (G0044)
* APT41 (G0096)
* Rocke (G0106)
* TeamTNT (G0139)
* APT28 (G0007)


In [15]:
groups_using_t1014

[{'object': IntrusionSet(type='intrusion-set', spec_version='2.1', id='intrusion-set--c5947e1c-1cbc-434c-94b8-27c7e3be0fff', created_by_ref='identity--c78cb6e5-0c4b-4611-8297-d1b8b55e40b5', created='2017-05-31T21:32:08.682Z', modified='2023-03-20T22:02:53.982Z', name='Winnti Group', description='[Winnti Group](https://attack.mitre.org/groups/G0044) is a threat group with Chinese origins that has been active since at least 2010. The group has heavily targeted the gaming industry, but it has also expanded the scope of its targeting.(Citation: Kaspersky Winnti April 2013)(Citation: Kaspersky Winnti June 2015)(Citation: Novetta Winnti April 2015) Some reporting suggests a number of other groups, including [Axiom](https://attack.mitre.org/groups/G0001), [APT17](https://attack.mitre.org/groups/G0025), and [Ke3chang](https://attack.mitre.org/groups/G0004), are closely linked to [Winnti Group](https://attack.mitre.org/groups/G0044).(Citation: 401 TRG Winnti Umbrella May 2018)', aliases=['Winnt

In [18]:
groups = mitre_attack_data.get_groups_using_technique("attack-pattern--0f20e3cb-245b-4a61-8a91-2d93f7cb0e9b")
groups = [x['object']['name'] for x in groups]
occurrence = len(groups)
occurrence

5

In [8]:
test = [{"name": name, "show": show} for name, show in list(zip(ics_platforms, [True]*len(enterprise_platforms)))]
test

[{'name': 'None', 'show': True}]

In [9]:
list(zip(enterprise_platforms, [True]*len(enterprise_platforms)))

[('Azure AD', True),
 ('Containers', True),
 ('macOS', True),
 ('Linux', True),
 ('Network', True),
 ('Google Workspace', True),
 ('SaaS', True),
 ('Office 365', True),
 ('PRE', True),
 ('Windows', True),
 ('IaaS', True)]